# Coeval kSZ — Exploration

Loads products from `data/products/`. Run `scripts/02_make_ksz_coeval_boxes.py` first.
Heavy functions live in `src/ksz_pipeline/coeval/`. This notebook is for plotting only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from ksz_pipeline.plotting.styles import PNG_STYLE, PDF_STYLE, save_pdf_png

## Load products

In [ ]:
Dl_data = np.load('data/products/ksz_Dl_coeval.npz')
reion   = np.load('data/products/coeval_reion.npz')
qperp   = np.load('data/products/qperp_power.npz', allow_pickle=True)
ell, Dl, sigma_Dl = Dl_data['ell'], Dl_data['Dl'], Dl_data['sigma_Dl']
z_re, xe, tau     = reion['z'], reion['xe'], reion['tau']
zs = qperp['z']
print(f'Loaded {len(zs)} redshift snapshots')

## D_ell vs lightcone (if available)

In [ ]:
try:
    lc = np.load('data/products/ksz_Dl_lightcone.npz')
    has_lc = True
except FileNotFoundError:
    has_lc = False

with mpl.rc_context(PNG_STYLE):
    fig, ax = plt.subplots(figsize=(10,7), constrained_layout=True)
    ax.errorbar(ell, Dl, yerr=sigma_Dl, fmt='o--', color='crimson',
                lw=1.5, ms=4, capsize=3, label=r'This work: Coeval Boxes $D_\ell$')
    if has_lc:
        ax.errorbar(lc['ell'], lc['Dl'], yerr=lc['Dl_err'], fmt='s-', color='darkblue',
                    lw=1.5, ms=4, capsize=3, label=r'This work: Lightcone $D_\ell$')
    ax.errorbar(3000, 1.1, yerr=[[0.7],[1.0]], fmt='s', ms=8,
                capsize=5, color='red', label='Reichardt+2021')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'Multipole $\ell$'); ax.set_ylabel(r'$D_\ell\ [\mu{\rm K}^2]$')
    ax.set_xlim(1e2,1e4); ax.set_ylim(1e-2,1e2)
    ax.legend(loc='upper left'); plt.show()

## P_{q_perp}(k) across redshifts

In [ ]:
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(zs)))
with mpl.rc_context(PNG_STYLE):
    fig, ax = plt.subplots(figsize=(9,7), constrained_layout=True)
    for i, (z_val, col) in enumerate(zip(zs, colors)):
        k = qperp[f'k_{i}']; P = qperp[f'Pq_{i}']; S = qperp[f'Pstd_{i}']
        ax.plot(k, P, color=col, lw=1.8)
        ax.fill_between(k, P-S, P+S, color=col, alpha=0.15)
    norm = plt.Normalize(vmin=zs.min(), vmax=zs.max())
    sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=ax, label=r'Redshift $z$')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$k\ [{\rm Mpc}^{-1}]$')
    ax.set_ylabel(r'$P_{q_\perp}(k)\ [{\rm cm}^2\,{\rm s}^{-2}\,{\rm Mpc}^3]$')
    plt.show()

## Reionization history + tau

In [ ]:
with mpl.rc_context(PNG_STYLE):
    fig, axes = plt.subplots(1,2, figsize=(14,6), constrained_layout=True)
    axes[0].plot(z_re, xe, lw=2, color='steelblue')
    axes[0].set_xlabel('Redshift z'); axes[0].set_ylabel(r'$\langle x_e \rangle$')
    axes[0].invert_xaxis()
    axes[1].plot(z_re, tau, lw=2, color='darkorange')
    axes[1].axhline(0.054, color='gray', ls='--', label='Planck 2018')
    axes[1].set_xlabel('Redshift z'); axes[1].set_ylabel(r'$\tau(z)$')
    axes[1].invert_xaxis(); axes[1].legend()
    plt.show()